# LAB 10 — Extractive Question Answering

## Notebook 1: Tokenizer and Span Alignment

Các mục 1–16 theo thứ tự trong W10.pdf. Mục tiêu là hiểu cách ghép question/context, ánh xạ character span sang token span và xử lý context dài.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
from qa_model import ROOT, DATA_DIR, RESULT_DIR, get_tokenizer, load_data, tokenize_examples, align_answer_to_tokens

data = load_data()
tokenizer = get_tokenizer()

D:\NLP_exercise\lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Dataset SQuAD

Mỗi mẫu gồm context, question, answers và answer_start. `answer_start` là character offset trong context. Dataset dưới đây là các split mẫu cố định từ SQuAD v1.1.

In [2]:
summary = pd.DataFrame([{"split":k,"examples":len(v)} for k,v in data.items()])
display(summary)
display(pd.DataFrame([{k: data["train"][0][k] for k in ["title","question","context","answers"]}]))

,split,examples
0,train,3000
1,validation,300
2,test,500


,title,question,context,answers
0,"St._John%27s,_Newfoundland_and_Labrador",What is the largest religion in St. John's?,"Predominantly Christian, the population of St....","[{'text': 'Christian', 'answer_start': 14}]"


## 10. Tokenization của QA

Pair input thường là `[CLS] question [SEP] context [SEP]`. `input_ids` là token ID, `attention_mask` đánh dấu token thật/padding; `token_type_ids` chỉ có ở một số model, không được mặc định mọi Transformer đều dùng.

In [6]:
question = "What is the capital of France?"
context = "The capital of France is Paris."
enc = tokenizer(question, context, return_offsets_mapping=True)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
rows=[]
for i,(tok,mask,off) in enumerate(zip(tokens,enc["attention_mask"],enc["offset_mapping"])):
    rows.append({"pair_token_index":i,"token":tok,"input_id":enc["input_ids"][i],"attention_mask":mask,"token_type_id":enc["token_type_ids"][i] if "token_type_ids" in enc else None,"offset":off})
display(pd.DataFrame(rows))

,pair_token_index,token,input_id,attention_mask,token_type_id,offset
0,0,[CLS],101,1,0,"(0, 0)"
1,1,what,2054,1,0,"(0, 4)"
2,2,is,2003,1,0,"(5, 7)"
3,3,the,1996,1,0,"(8, 11)"
4,4,capital,3007,1,0,"(12, 19)"
5,5,of,1997,1,0,"(20, 22)"
6,6,france,2605,1,0,"(23, 29)"
7,7,?,1029,1,0,"(29, 30)"
8,8,[SEP],102,1,0,"(0, 0)"
9,9,the,1996,1,1,"(0, 3)"


## 11. Offset Mapping

Dataset đánh dấu đáp án bằng ký tự, còn mô hình dự đoán token position. Offset mapping nối từng token context với khoảng ký tự `[start,end)` trong văn bản gốc.

In [7]:
seq_ids = enc.sequence_ids()
rows=[{"token":tokens[i],"sequence_id":seq_ids[i],"offset":enc["offset_mapping"][i]} for i in range(len(tokens))]
display(pd.DataFrame(rows))

,token,sequence_id,offset
0,[CLS],NaN,"(0, 0)"
1,what,0.0,"(0, 4)"
2,is,0.0,"(5, 7)"
3,the,0.0,"(8, 11)"
4,capital,0.0,"(12, 19)"
5,of,0.0,"(20, 22)"
6,france,0.0,"(23, 29)"
7,?,0.0,"(29, 30)"
8,[SEP],NaN,"(0, 0)"
9,the,1.0,"(0, 3)"


## 14. Experiment 1 — Context Length

So sánh max_length 128/256/384 trong khi giữ train sample, stride, epoch và model cố định. Điểm số không nhất thiết tăng đều khi tăng context length.

In [10]:
context_results = pd.read_csv(RESULT_DIR/"context_length_results.csv")
display(context_results.round(3))

,max_length,stride,train_examples,epochs,training_seconds,test_em,test_f1
0,128,64,600,1,17.779,0.006,0.044
1,256,64,600,1,16.151,0.016,0.082
2,384,64,600,1,18.999,0.014,0.086


### Thảo luận và trả lời câu hỏi

**Tăng context length có luôn cải thiện QA không? Không.** Với cùng 600 ví dụ train, stride 64 và một epoch, kết quả test là:

| Max length | EM | F1 | Thời gian đo (giây) |
|---:|---:|---:|---:|
| 128 | 0.006 | 0.044 | 17.8 |
| 256 | 0.016 | 0.082 | 16.2 |
| 384 | 0.014 | 0.086 | 19.0 |

Từ 128 lên 256, cả EM và F1 đều tăng. Từ 256 lên 384, F1 tăng nhẹ nhưng EM giảm; 384 cũng không có EM cao nhất và mất nhiều thời gian hơn 256 trong lượt chạy này. Context dài có thể giữ lại thêm bằng chứng, nhưng không bảo đảm span chính xác hơn và làm tăng lượng đầu vào. Kết quả chỉ dựa trên một lượt chạy với tập mẫu nhỏ, nên không kết luận cấu hình 256 luôn tốt hơn.

## 15. Core Implementation

Dùng tokenizer trực tiếp với `truncation="only_second"`, `stride`, `return_overflowing_tokens=True` và `return_offsets_mapping=True`; không chỉ gọi một QA pipeline có sẵn.

In [11]:
core = tokenizer(question, context, truncation="only_second", max_length=384, stride=128, return_overflowing_tokens=True, return_offsets_mapping=True)
print("Keys:",list(core.keys()))
print("input_ids (first 16):",core["input_ids"][0][:16])
print("attention_mask (first 16):",core["attention_mask"][0][:16])
print("offset_mapping (first 16):",core["offset_mapping"][0][:16])
print("overflow_to_sample_mapping:",core["overflow_to_sample_mapping"])

Keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
input_ids (first 16): [101, 2054, 2003, 1996, 3007, 1997, 2605, 1029, 102, 1996, 3007, 1997, 2605, 2003, 3000, 1012]
attention_mask (first 16): [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
offset_mapping (first 16): [(0, 0), (0, 4), (5, 7), (8, 11), (12, 19), (20, 22), (23, 29), (29, 30), (0, 0), (0, 3), (4, 11), (12, 14), (15, 21), (22, 24), (25, 30), (30, 31)]
overflow_to_sample_mapping: [0]


## 16. Answer Span Alignment

Chuyển `[answer_start, answer_end)` sang token start/end bằng offset mapping và sequence ID. Kiểm tra answer ở giữa token, bị truncate hoặc nằm ngoài một window.

In [12]:
offsets=list(core["offset_mapping"][0]); seq=core.sequence_ids(0)
print("Paris alignment:",align_answer_to_tokens(offsets,seq,char_start,answer))
print("Outside-window alignment:",align_answer_to_tokens(offsets,seq,len(context)+100,"missing"))

Paris alignment: (14, 14)
Outside-window alignment: (0, 0)
